# Report prose vs the signed conviction

Every analyst emits two things about the same judgement: a **number** (direction ×
conviction) and **120–250 words of prose**. The number is what every IC in this project
grades. The prose is what actually crosses the boundary to an LLM PM.

So there are two questions, and they are not the same:

| | question | if the answer is no… |
|---|---|---|
| **alignment** | does the prose say what the number says? | the report and the header disagree — the PM is reading one thing and scoring another |
| **redundancy** | does the prose carry information the number does not? | the layer is paying for prose that a float could replace |

A report can be perfectly aligned and perfectly redundant at the same time. That is the
outcome worth being able to detect, because it is the one that makes the prose channel
expensive and pointless.

**The prose signal is a lexicon count and stays one.** `(accel hits − decel hits) /
total`, scaled by the header's own conviction so both live on the same axis. That is not
reading comprehension, and `report_quality`'s own docstring is right that an LLM-judge
pass is the deeper layer and adds a judge plus a confound. Read a null here as *the coarse
check found nothing*, not as *the prose is empty* — three earlier lexical rates in this
project turned out to be artifacts of the check rather than of the reports.

In [ ]:
import sys, os, glob, math, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("..")); sys.path.insert(0, ".")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import _viz as V; V.use_style()

from src.layered.evaluation.runs import load_run
from src.layered.evaluation import ic_diagnostics as D
from src.layered.evaluation import attribution as A

BOARD, ARM = "../reports/hk", "anon_cue"
ARMS = ["none", "anon_cue", "anon_full", "plain"]
LABEL = {"none": "none\n(numbers only)", "anon_cue": "anon_cue\n(driver extract)",
         "anon_full": "anon_full\n(whole statement)", "plain": "plain\n(LEAK ARM)"}

runs = {}
for f in sorted(glob.glob(f"{BOARD}/*.jsonl")):
    st = os.path.basename(f)[:-6]
    for a in ARMS:
        if st.endswith("_" + a):
            runs[(st[:-(len(a)+1)], a)] = load_run(f); break
DRIVERS = sorted({d for d, _ in runs})
IC = {(d, a): D.rank_ic(*[D.align(runs[(d,a)].signed.dropna(), runs[(d,a)].level.dropna())[c]
                          for c in ("s","y")])[1] for d, a in runs}
print(f"{len(runs)} legs · {len(DRIVERS)} analysts")

In [ ]:
from src.layered.evaluation.report_quality import _ACCEL, _DECEL, _contains, evaluate_run

def prose_lean(text):
    if not text: return 0.0
    up, dn = len(_contains(text, _ACCEL)), len(_contains(text, _DECEL))
    return 0.0 if up + dn == 0 else (up - dn) / (up + dn)

def leg_stats(driver, arm):
    r = runs[(driver, arm)]
    v = r.views; live = ~v["degraded"]
    lean = v.loc[live, "report"].map(prose_lean)
    prose = (lean * v.loc[live, "conviction"]).dropna()
    header = r.signed.dropna()
    lvl = r.level.dropna()
    ev = D.rank_ic
    a_h = D.align(header, lvl); a_p = D.align(prose, lvl)
    n_h, ic_h, t_h = ev(a_h["s"], a_h["y"]); n_p, ic_p, t_p = ev(a_p["s"], a_p["y"])
    both = pd.concat([header.rename("h"), prose.rename("p")], axis=1).dropna()
    nz = both[(both["h"] != 0) & (both["p"] != 0)]
    q = evaluate_run(f"{BOARD}/{driver}_{arm}.jsonl", driver=driver)
    words = v.loc[live, "report"].map(lambda x: len((x or "").split()))
    return {"driver": driver, "arm": arm, "n": n_h,
            "ic_header": ic_h, "ic_prose": ic_p, "delta": ic_p - ic_h,
            "t_header": t_h, "t_prose": t_p,
            "sign_agree": float((np.sign(nz["h"]) == np.sign(nz["p"])).mean()) if len(nz) else np.nan,
            "dir_consistent": q.get("dir_consistent"),
            "prose_mute": float((lean == 0).mean()),
            "med_words": float(words.median()),
            "in_contract": float(((words >= 120) & (words <= 250)).mean())}

TX = pd.DataFrame([leg_stats(d, a) for d in DRIVERS for a in ARMS])
TX.round(3).head()

## 1 — Alignment: does the prose agree with the number?

Two measures of the same thing, deliberately shown together because they disagree and the
gap is informative.

**`dir_consistent`** (left) is `report_quality`'s own check, and it counts a *flat* header
or a *mute* prose lean as consistent — so it is generous by construction.
**`sign_agree`** (right) is strict: restricted to meetings where both the prose and the
header took a side, how often did they take the same one? 0.5 is a coin flip.

In [ ]:
t = TX[TX["arm"] == ARM].sort_values("sign_agree")
fig, (a1, a2) = V.paired_rank(list(t["driver"]), t["dir_consistent"].values, t["sign_agree"].values,
    left_label="dir_consistent  (generous: flat/mute counts as agreeing)",
    right_label="strict sign agreement, both sides taken",
    left_ref=None, right_ref=0.5, figsize=(12.4, 5.2), sort_by="right")
V.title(a1, "The prose and the header agree on 72% of two-sided calls, not the 87% the lenient check reports",
        f"arm = {ARM}. Left counts flat/mute as agreeing; right does not. Dashed line is a coin flip.")
plt.show()

print("median dir_consistent %.3f   median strict sign agreement %.3f"
      % (t["dir_consistent"].median(), t["sign_agree"].median()))
print("The gap is the flat/mute cases the lenient check forgives.")

## 2 — Redundancy: does the prose carry anything the number does not?

Both signals scored against the same outcome by the same evaluator. If the prose bar sits
below the header bar everywhere, the prose is a noisier restatement — aligned but
redundant, which is the expensive outcome.

Built in the shape of `ICEvaluator.calibration_split`, which asks the same question one
level down (what does conviction *magnitude* add over direction alone).

In [ ]:
t = TX[TX["arm"] == ARM].sort_values("ic_header")
y = np.arange(len(t))
fig, ax = plt.subplots(figsize=(10.4, 5.6))
ax.hlines(y, t["ic_prose"], t["ic_header"], color=V.GRID, lw=2.2, zorder=2)
ax.scatter(t["ic_header"], y, s=88, color=V.POS, zorder=4, edgecolor=V.SURFACE,
           linewidth=1.3, label="header  (direction × conviction)")
ax.scatter(t["ic_prose"], y, s=88, color=V.CAT[1], zorder=4, edgecolor=V.SURFACE,
           linewidth=1.3, label="prose  (lexicon lean × conviction)")
ax.axvline(0, color=V.AXIS, lw=1.0, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(t["driver"])
ax.set_xlabel("rank IC against the driver's own next-release change")
ax.legend(loc="lower right", ncols=1)
V.title(ax, "On nine of eleven analysts the prose is a weaker signal than the number",
        f"arm = {ARM}. Each row is one analyst. The two exceptions are analysts whose header "
        f"IC is negative —\nthere the prose is less wrong, which is not the same as being right.")
V.grid(ax, "x"); plt.show()

d = TX[TX["arm"] == ARM]["delta"]
print(f"prose − header IC:  median {d.median():+.3f}   best {d.max():+.3f}   worst {d.min():+.3f}")
print(f"prose beat the header on {int((d > 0).sum())}/{len(d)} analysts")

## 3 — Does the text arm change any of this?

The features are byte-identical across arms; only the text block differs. So a shift here
is caused by the text channel — and it can be read without the IC moving at all.

In [ ]:
piv = TX.pivot_table(index="driver", columns="arm", values="sign_agree")[ARMS]
fig, ax = plt.subplots(figsize=(8.6, 5.0))
im = ax.imshow(piv.values, cmap=V.SEQ, aspect="auto", vmin=0.4, vmax=0.75)
ax.set_xticks(range(len(ARMS))); ax.set_xticklabels([LABEL[a] for a in ARMS], fontsize=8.5)
ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8.5,
                color="white" if v > 0.62 else V.INK)
for sp in ax.spines.values(): sp.set_visible(False)
cb = fig.colorbar(im, ax=ax, pad=.02, fraction=.03); cb.ax.tick_params(labelsize=8)
V.title(ax, "Prose–header agreement barely moves with the text arm",
        "Strict sign agreement. 0.50 would be a coin flip.")
plt.show()
display(TX.groupby("arm")[["sign_agree","dir_consistent","delta","med_words","in_contract"]]
        .mean().reindex(ARMS).round(3))

## 4 — Length: the reports overrun their contract, and the PM pays for it

The output contract asks for 120–250 words. `render_brief` concatenates seven of these into
one brief, so the overrun is not cosmetic — it decides whether `--max-report-words` binds.

In [ ]:
fig, ax = plt.subplots(figsize=(10.8, 4.4))
for i, arm in enumerate(ARMS):
    w = []
    for d in DRIVERS:
        v = runs[(d, arm)].views
        w += list(v.loc[~v["degraded"], "report"].map(lambda x: len((x or "").split())))
    w = np.array(w)
    x = np.random.default_rng(i).normal(i, 0.075, len(w))
    ax.scatter(x, w, s=5, color=V.CAT[i], alpha=.28, zorder=3, linewidth=0)
    ax.hlines(np.median(w), i-.3, i+.3, color=V.INK, lw=2.2, zorder=5)
    ax.annotate(f"median {np.median(w):.0f}", (i, np.median(w)), textcoords="offset points",
                xytext=(24, -3), fontsize=8.8, color=V.INK)
ax.axhspan(120, 250, color=V.BAND, zorder=1)
ax.annotate("the 120–250 word contract", xy=(3.42, 250), xytext=(0, 5),
            textcoords="offset points", fontsize=8.8, color=V.MUTED, ha="right")
ax.set_xticks(range(4)); ax.set_xticklabels([LABEL[a] for a in ARMS], fontsize=9)
ax.set_ylabel("words per report")
V.title(ax, "Every arm overruns the word contract, and the leak arm overruns it most",
        "One dot per report. Seven of these make one PM brief.")
V.grid(ax, "y"); plt.show()
print(TX.groupby("arm")["in_contract"].mean().reindex(ARMS).round(3).to_string())

## What this can and cannot say

- **The prose signal is a lexicon count.** A null is the coarse check finding nothing, not
  proof the prose is empty. The honest deeper test is an LLM judge, which adds a judge and
  a confound and has not been run.
- **`dir_consistent` is deliberately lenient** — flat headers and mute prose count as
  agreeing — which is why it is shown beside the strict measure rather than instead of it.
- **`plain` is the leak arm.** Its numbers appear here for completeness of the arm
  comparison and are not analyst results.
- Alignment and redundancy are independent. High alignment with negative delta — which is
  what this board shows — means the prose faithfully restates a number that already carried
  the information.